In [1]:
import json
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
# To install: pip install tavily-python
from langchain_tavily import TavilySearch
#stategraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
## custom tool function
from langchain_core.tools import tool
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver

tavily_search = TavilySearch(max_result=2)
tavily_search.invoke("what is langgraph")

{'query': 'what is langgraph',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ibm.com/think/topics/langgraph',
   'title': 'What is LangGraph? - IBM',
   'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. The following example can offer a clearer understanding of LangGraph: Think about these graph-based architectures as a powerful configurable map, a “Super-Map.” Users can envision the AI workflow as being “The Navigator” of this “Super-Map.” Finally, in this example, the user is “The Cartographer.” In this sense, the n

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [6]:
question = "any idea about healthy breakfast"

llm.invoke(question).content

'A healthy breakfast is essential to kick-start your day. Here are some ideas:\n\n**Why is breakfast important?**\n\n1. Boosts energy and alertness\n2. Supports weight management\n3. Helps maintain concentration and focus\n4. Supports healthy digestion and metabolism\n5. Can help lower the risk of chronic diseases, such as heart disease and diabetes\n\n**Healthy Breakfast Ideas:**\n\n1. **Oatmeal with fruits and nuts**: Steel-cut oats or rolled oats with fresh fruits, nuts, and a drizzle of honey.\n2. **Avocado toast**: Whole-grain toast topped with mashed avocado, eggs, and cherry tomatoes.\n3. **Greek yogurt with berries and granola**: Greek yogurt with fresh berries, granola, and a sprinkle of cinnamon.\n4. **Veggie omelette**: Whisked eggs with sautéed vegetables, such as bell peppers, onions, and mushrooms.\n5. **Smoothie bowl**: Blend of Greek yogurt, frozen fruits, and spinach, topped with granola, nuts, and seeds.\n6. **Whole-grain cereal with milk and banana**: Whole-grain cer

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a dr. paul saladino , "Carnivore MD", advocating for animal-based nutrition and challenging plant-centric dietary dogma. 
            1.{first_instruction}
            """
        ),
        MessagesPlaceholder(variable_name="messages"),
        ("system", "Answer the user's question above using the requirement format.")
    
    ]
)

In [5]:
first_response = prompt_template.partial(
    first_instruction="provide a detailes 100 word answer"
)

temp_chain = first_response | llm

In [7]:
response = temp_chain.invoke({"messages":[HumanMessage(content=question)]})

In [8]:
response.content

'As the "Carnivore MD", I recommend a breakfast rich in animal-derived nutrients. Consider a plate of eggs, bacon, and beef, providing protein, vitamins, and minerals. Alternatively, a bowl of beef broth with organ meats like liver and kidney offers a nutrient-dense start. Avoid plant-based options like oatmeal, fruit, and whole grains, which can be high in carbohydrates and low in essential nutrients. A carnivore-based breakfast supports optimal energy, satiety, and overall health, setting you up for a day of vitality and wellness, with a focus on animal-based nutrition for a healthy and balanced diet always.'

In [10]:
# create reflection sehema or class

class Reflection(BaseModel):
    missing: str = Field(description="what information is missing")
    superfluous: str = Field(description="what information is necessary")

class AnswerQuestion(BaseModel):
    answer: str = Field(description="main responset to the question")
    reflection: Reflection = Field(description="self-critique of the answer")
    search_queries: List[str] = Field(description="query for additional research")

In [11]:
initial_chain = first_response | llm.bind_tools(tools=[AnswerQuestion])

In [14]:
response = initial_chain.invoke({"messages":[HumanMessage(question)]})

response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'f3n8ej278', 'function': {'arguments': '{"answer":"A healthy breakfast should prioritize animal-based foods like eggs, meat, and fish, which provide essential nutrients and satiety. Consider a breakfast consisting of scrambled eggs with bacon or sausage, or a bowl of beef or chicken broth with some added fat. Avoid plant-based options like grains, fruits, and vegetables, which are high in carbohydrates and low in nutrients. This approach will help regulate blood sugar, promote weight loss, and support overall health.","reflection":{"missing":"specific nutritional information","superfluous":"none"},"search_queries":["animal-based breakfast ideas","plant-based breakfast drawbacks"]}', 'name': 'AnswerQuestion'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 348, 'total_tokens': 485, 'completion_time': 0.492768263, 'completion_tokens_details': None, 'prompt_time': 0.0751031

In [16]:
response_list = []
response_list.append(HumanMessage(content=question))
response_list.append(response)

tool_call = response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
search_queries

['animal-based breakfast ideas', 'plant-based breakfast drawbacks']

In [ ]:
# create a function for search queries
import json
from typing import List, Dict, Any
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage

def execute_tools(state:List[BaseMessage]) -> List[BaseMessage]:
    last_ai_messgae